<a href="https://colab.research.google.com/github/krhoden725-a11y/Final-React-Native-Assingment/blob/main/easy_stats_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Easy Stats — Hypothesis Testing for Marketing Analytics

**FreshBrew Coffee: The Q4 Board Meeting**

FreshBrew Coffee invested heavily this quarter: a redesigned mobile app, new POS terminals, a loyalty program, digital ad campaigns, and a pricing experiment.

The board wants **statistical evidence** — not opinions, not vibes.

---

### Your Files

Upload these 5 files to Colab:

| File | What it contains |
|------|------------------|
| `easy_stats.py` | The testing functions |
| `freshbrew_survey.csv` | Customer survey (NPS, satisfaction) |
| `freshbrew_operations.csv` | Store data (wait times, churn, revenue) |
| `freshbrew_digital.csv` | Website sessions (clicks, purchases, devices) |
| `freshbrew_transactions.csv` | Individual purchases (spend, membership) |

---
## Setup

Run this cell first. It loads the easy_stats functions.

In [8]:
from easy_stats import *
import pandas as pd

---
# PART 1: Testing Means
---

## Question 1: Has NPS Changed After the App Redesign?

> The CEO: *"Our NPS has been 72 for three straight years. We redesigned the mobile app. **Has NPS changed?** I don't know if it went up or down — I just want to know if it moved."*

This is a **one-sample t-test** (comparing one group's mean to a known benchmark).

It is **two-tailed** because the CEO is not predicting a direction — just asking "did it change?"

### Step 1: Load the data

In [16]:
survey = pd.read_csv('freshbrew_survey.csv')
survey.head()

,customer_id,segment,nps_score,satisfaction,monthly_visits,age_group,region
0,1001,App_User,77,9.5,6,25-34,Midwest
1,1002,Non_App_User,65,7.6,2,35-44,Southeast
2,1003,Non_App_User,85,5.9,3,35-44,Southeast
3,1004,Non_App_User,97,6.8,4,35-44,Southeast
4,1005,App_User,47,8.0,3,18-24,Midwest


### Step 2: Pick the column we need

We want the `nps_score` column. In pandas, we grab a column with `df['column_name']`.

In [17]:
nps_scores = survey['nps_score']

print("Number of responses:", len(nps_scores))
print("Average NPS:", round(nps_scores.mean(), 1))

Number of responses: 500
Average NPS: 70.7


### Step 3: Run the test

We pass the column to `one_sample_t()` and set `H0=72` (the benchmark).

In [18]:
one_sample_t(series=nps_scores, H0=72)


ONE-SAMPLE T-TEST
H0: mean = 72
H1: mean != 72

Sample mean:  70.742
Sample std:   21.2205
Sample size:  500
t-statistic:  -1.3256
p-value:      0.1856

DECISION: Fail to reject H0
  p-value 0.1856 >= 0.05
  Not enough evidence that mean != 72



{'t_stat': np.float64(-1.3255947850513325),
 'p_value': np.float64(0.1855804980941464),
 'mean': np.float64(70.742),
 'std': 21.2204630511261,
 'n': 500,
 'reject_H0': np.False_}

### Your Turn

1. What is the null hypothesis in plain English?
2. Look at the p-value. Is it less than 0.05?
3. What do you tell the CEO — has NPS changed or not?

---
## Question 2: Did New POS Terminals Reduce Wait Times?

> *"We spent \$180K on new POS terminals for the drive-through. The historical average wait was 4.5 minutes. **Are the new terminals faster?**"*

This is still a **one-sample t-test**, but now we need to **filter the data first** — we only want drive-through stores that have the new terminals.

It is **left-tailed** (`alt='less'`) because we specifically predict wait times went *down*.

### Step 1: Load the data

In [26]:
operations = pd.read_csv('freshbrew_operations.csv')
operations.head()

,store_id,store_type,has_new_pos,retention_campaign,region,wait_time_min,monthly_churn_pct,avg_ticket_usd,monthly_revenue,instagram_followers
0,1,Drive_Through,False,Control,Midwest,5.19,15.6,6.35,14017.0,2271
1,2,Drive_Through,True,Control,Northeast,6.52,16.2,7.12,20994.0,2736
2,3,Drive_Through,False,Campaign,Southeast,4.98,11.7,8.08,14068.0,1969
3,4,Drive_Through,True,Control,Northeast,2.93,11.0,5.19,17115.0,4732
4,5,Cafe,True,Control,Southeast,7.19,12.7,6.20,13068.0,2382


In [15]:
!mv freshbrew_operations.numbers freshbrew_operations.csv

mv: cannot stat 'freshbrew_operations.numbers': No such file or directory


Let's see what store types and POS options are in the data:

In [ ]:
print("Store types:")
print(operations['store_type'].value_counts())
print()
print("Has new POS:")
print(operations['has_new_pos'].value_counts())

### Step 2: Filter the data

We only want **drive-through** stores that **have the new POS**. We use conditions inside `df[...]` to filter:

In [27]:
# Filter: drive-through stores AND new POS terminals
dt_new_pos = operations[(operations['store_type'] == 'Drive_Through') & (operations['has_new_pos'] == True)]

print("Stores matching our filter:", len(dt_new_pos))

Stores matching our filter: 88


### Step 3: Pick the column and run the test

In [28]:
wait_times = dt_new_pos['wait_time_min']

print("Average wait time:", round(wait_times.mean(), 2), "minutes")

Average wait time: 3.87 minutes


In [29]:
# Test: is the mean wait time LESS than 4.5 minutes?
one_sample_t(series=wait_times, H0=4.5, alt='less')


ONE-SAMPLE T-TEST
H0: mean = 4.5
H1: mean < 4.5

Sample mean:  3.8693
Sample std:   1.6096
Sample size:  88
t-statistic:  -3.6757
p-value:      0.0002

DECISION: Reject H0
  p-value 0.0002 < 0.05
  Evidence that mean < 4.5



{'t_stat': np.float64(-3.675651745978996),
 'p_value': np.float64(0.00020482756237018846),
 'mean': np.float64(3.8693181818181817),
 'std': 1.6095975048072466,
 'n': 88,
 'reject_H0': np.True_}

### Your Turn

1. Why did we use `alt='less'` instead of the default two-sided?
2. Should the CFO approve new POS for 50 more stores? What is your evidence?
3. **Try it:** Test whether **Cafe** stores with new POS have wait times below their historical average of **6.0 minutes**. Change the filter and the H0.

In [ ]:
# YOUR CODE HERE
# Hint: change 'Drive_Through' to 'Cafe' and H0 to 6.0


---
## Question 3: Is Our Email Open Rate Beating the Industry? (Plug-in Numbers)

> *"The 2024 Mailchimp benchmark for food & beverage is 21.0% open rate. Our marketing director pulled stats from 500 campaign sends: mean = 23.8%, std = 11.5%. Are we beating the benchmark?"*

Sometimes you don't have the raw data — just summary statistics from a report. You can **plug in the numbers** directly.

**Right-tailed** (`alt='greater'`) because the director claims we are *above* the benchmark.

In [ ]:
# No data to load — just plug in the numbers from the report
one_sample_t(sample_mean=23.8, sample_std=11.5, n=500, H0=21.0, alt='greater')

### Your Turn

1. What would change if the director said "we're *different* from benchmark" instead of "we *beat* it"?
2. Is our open rate significantly above the industry benchmark?

---
## Question 4: Do Loyalty Members Spend More?

> *"The CFO keeps saying loyalty members would spend a lot **anyway**. Do loyalty members actually spend more per visit?"*

This is a **two-sample t-test** — we are comparing the means of two groups.

**Right-tailed** (`alt='greater'`) because we predict members spend *more*.

### Step 1: Load the data

In [31]:
transactions = pd.read_csv('freshbrew_transactions.csv')
transactions.head()

,transaction_id,store_id,membership,spend_usd,items_purchased,hour_of_day,day_of_week,payment_method
0,5001,3,Non_Member,5.45,5,8,Thu,App
1,5002,6,Non_Member,6.07,1,13,Thu,Card
2,5003,32,Non_Member,6.26,2,9,Sat,Cash
3,5004,49,Non_Member,7.12,2,20,Mon,Card
4,5005,45,Non_Member,6.41,2,8,Mon,Card


### Step 2: Split into two groups

For a two-sample test, we need to create **two separate series** — one for each group.

In [32]:
# Filter to get each group, then pick the spend column
members     = transactions[transactions['membership'] == 'Member']['spend_usd']
non_members = transactions[transactions['membership'] == 'Non_Member']['spend_usd']

print("Members:     n =", len(members),     "  mean = $" + str(round(members.mean(), 2)))
print("Non-members: n =", len(non_members), "  mean = $" + str(round(non_members.mean(), 2)))

Members:     n = 772   mean = $6.81
Non-members: n = 1228   mean = $5.4


### Step 3: Run the test

In [33]:
# Do members (group 1) spend MORE than non-members (group 2)?
two_sample_t(series1=members, series2=non_members, alt='greater')


TWO-SAMPLE T-TEST
H0: mean1 - mean2 = 0
H1: mean1 - mean2 > 0

Group 1:  mean = 6.8062  std = 2.4166  n = 772
Group 2:  mean = 5.3996  std = 2.0033  n = 1228
Difference:  1.4066
t-statistic: 13.5143
p-value:     0.0

DECISION: Reject H0
  p-value 0.0 < 0.05
  Evidence that mean1 - mean2 > 0



{'t_stat': np.float64(13.514267330375015),
 'p_value': np.float64(1.561759322873301e-39),
 'mean1': np.float64(6.806217616580311),
 'mean2': np.float64(5.399617263843648),
 'diff': np.float64(1.406600352736663),
 'reject_H0': np.True_}

### Your Turn

1. Should the VP get the $500K loyalty program budget? What is your evidence?
2. **Try it:** Do members also buy **more items** per visit? Use the `items_purchased` column and run a right-tailed two-sample t-test.

In [34]:
# # Filter to get each group, then pick the spend column
members     = transactions[transactions['membership'] == 'Member']['items_purchased']
non_members = transactions[transactions['membership'] == 'Non_Member']['items_purchased']


---
## Question 5: Did the Retention Campaign Cut Churn?

> *"We ran a 90-day retention campaign in half our stores. The other half got nothing. **Did the campaign reduce churn?**"*

Another **two-sample t-test**. **Left-tailed** because we predict campaign stores have *lower* churn.

In [35]:
# operations was already loaded above
campaign = operations[operations['retention_campaign'] == 'Campaign']['monthly_churn_pct']
control  = operations[operations['retention_campaign'] == 'Control']['monthly_churn_pct']

print("Campaign stores: n =", len(campaign), " mean churn =", round(campaign.mean(), 1), "%")
print("Control stores:  n =", len(control),  " mean churn =", round(control.mean(), 1), "%")

Campaign stores: n = 248  mean churn = 11.1 %
Control stores:  n = 252  mean churn = 13.7 %


In [36]:
# Does campaign (group 1) have LESS churn than control (group 2)?
two_sample_t(series1=campaign, series2=control, alt='less')


TWO-SAMPLE T-TEST
H0: mean1 - mean2 = 0
H1: mean1 - mean2 < 0

Group 1:  mean = 11.052  std = 4.9947  n = 248
Group 2:  mean = 13.6857  std = 5.9187  n = 252
Difference:  -2.6337
t-statistic: -5.3804
p-value:     0.0

DECISION: Reject H0
  p-value 0.0 < 0.05
  Evidence that mean1 - mean2 < 0



{'t_stat': np.float64(-5.380425792344719),
 'p_value': np.float64(5.784143927171213e-08),
 'mean1': np.float64(11.052016129032257),
 'mean2': np.float64(13.685714285714287),
 'diff': np.float64(-2.63369815668203),
 'reject_H0': np.True_}

### Your Turn

1. Is the churn reduction statistically significant?
2. Should FreshBrew roll out the campaign to all stores?

---
## Question 6: Landing Page A/B Test (Plug-in Numbers)

> *"The growth team ran an A/B test. Control: n=3000, mean=42.0 sec, std=18.5. New page: n=3000, mean=42.8 sec, std=21.0. Are they different?"*

Again, no raw data — just plug in the summary stats. **Two-tailed** — just asking "are they different?"

In [37]:
two_sample_t(mean1=42.0, std1=18.5, n1=3000,
             mean2=42.8, std2=21.0, n2=3000)


TWO-SAMPLE T-TEST
H0: mean1 - mean2 = 0
H1: mean1 - mean2 != 0

Group 1:  mean = 42.0  std = 18.5  n = 3000
Group 2:  mean = 42.8  std = 21.0  n = 3000
Difference:  -0.8
t-statistic: -1.5657
p-value:     0.1175

DECISION: Fail to reject H0
  p-value 0.1175 >= 0.05
  Not enough evidence that mean1 - mean2 != 0



{'t_stat': np.float64(-1.5656706578974202),
 'p_value': np.float64(0.1174793307795345),
 'mean1': 42.0,
 'mean2': 42.8,
 'diff': -0.7999999999999972,
 'reject_H0': np.False_}

### Your Turn

1. The new page had 0.8 seconds more engagement. Why is that not statistically significant?
2. What should the growth team do next?

---
# PART 2: Testing Proportions

> *"Let's shift to the digital metrics. These are proportions — click rates, conversion rates — not dollar amounts."*

---

## Question 7: Is Our Google Ads CTR Above Benchmark?

> *"The food & beverage benchmark CTR is 4.7%. Is FreshBrew's Google Ads CTR actually higher?"*

This is a **one-sample proportion test**. The data is 0s and 1s (clicked or didn't click).

**Right-tailed** because we are asking "is it *higher*?"

### Step 1: Load and filter

In [38]:
digital = pd.read_csv('freshbrew_digital.csv')
digital.head()

,session_id,device,source,loyalty_member,clicked_ad,purchased,added_to_cart,abandoned_cart,time_on_site_sec,pages_viewed,unsubscribed
0,1,Desktop,Google_Ads,False,0,0,0,0,4.6,1,0
1,2,Tablet,Email,False,0,0,1,0,65.6,2,0
2,3,Mobile,Google_Ads,True,0,0,0,0,3.6,6,0
3,4,Mobile,Google_Ads,False,0,0,1,1,14.4,3,0
4,5,Desktop,Instagram,True,0,0,0,0,13.2,4,0


In [39]:
# What sources do we have?
print(digital['source'].value_counts())

source
Organic       2574
Google_Ads    2529
Email         1940
Instagram     1552
Direct        1405
Name: count, dtype: int64


In [40]:
# Filter to Google Ads sessions only
google_ads = digital[digital['source'] == 'Google_Ads']

print("Google Ads sessions:", len(google_ads))
print("Clicks:", google_ads['clicked_ad'].sum())

Google Ads sessions: 2529
Clicks: 127


### Step 2: Pick the column and test

The `clicked_ad` column is already 0s and 1s — perfect for a proportion test.

In [41]:
one_sample_z_prop(series=google_ads['clicked_ad'], H0=0.047, alt='greater')


ONE-SAMPLE Z-TEST FOR PROPORTION
H0: proportion = 0.047
H1: proportion > 0.047

Successes:          127
Sample size:        2529
Sample proportion:  0.0502
z-statistic:        0.7645
p-value:            0.2223

DECISION: Fail to reject H0
  p-value 0.2223 >= 0.05
  Not enough evidence that proportion > 0.047



{'z_stat': np.float64(0.7645299829726022),
 'p_value': np.float64(0.22227573804687223),
 'p_hat': np.float64(0.05021747726374061),
 'n': 2529,
 'reject_H0': np.False_}

### Your Turn

1. Is FreshBrew's Google Ads CTR significantly above the 4.7% benchmark?
2. **Try it:** Test whether **Instagram** ads (`source == 'Instagram'`) beat the Instagram benchmark of **2.1%**.

In [47]:

# Filter to Google Ads sessions only
google_ads = digital[digital['source'] == 'Instagram']

print("IG Ads sessions:", len(google_ads))
print("Clicks:", google_ads['clicked_ad'].sum())

one_sample_z_prop(series=google_ads['clicked_ad'], H0=0.021, alt='greater')


IG Ads sessions: 1552
Clicks: 54

ONE-SAMPLE Z-TEST FOR PROPORTION
H0: proportion = 0.021
H1: proportion > 0.021

Successes:          54
Sample size:        1552
Sample proportion:  0.0348
z-statistic:        3.7899
p-value:            0.0001

DECISION: Reject H0
  p-value 0.0001 < 0.05
  Evidence that proportion > 0.021



{'z_stat': np.float64(3.7899130331377235),
 'p_value': np.float64(7.535001848941513e-05),
 'p_hat': np.float64(0.03479381443298969),
 'n': 1552,
 'reject_H0': np.True_}

---
## Question 8: Did More Emails Increase Unsubscribes? (Plug-in Numbers)

> *"We increased email frequency from 2 to 4 per week. Historical unsubscribe rate is 0.8%. A report says we had 18 unsubscribes out of 1,500 sends. Did unsubscribes go up?"*

When someone hands you a report with just count and total, you can plug them in directly.

**Right-tailed** — we are worried it went *up*.

In [48]:
# Plug in the numbers from the report
one_sample_z_prop(count=18, n=1500, H0=0.008, alt='greater')


ONE-SAMPLE Z-TEST FOR PROPORTION
H0: proportion = 0.008
H1: proportion > 0.008

Successes:          18
Sample size:        1500
Sample proportion:  0.012
z-statistic:        1.739
p-value:            0.041

DECISION: Reject H0
  p-value 0.041 < 0.05
  Evidence that proportion > 0.008



{'z_stat': np.float64(1.739020859100631),
 'p_value': np.float64(0.04101554695946932),
 'p_hat': 0.012,
 'n': 1500,
 'reject_H0': np.True_}

We can also verify with the actual data in the digital dataset:

In [ ]:
# Filter to email sessions, then check the unsubscribed column
email_sessions = digital[digital['source'] == 'Email']

print("Email sessions:", len(email_sessions))
print("Unsubscribes:", email_sessions['unsubscribed'].sum())
print()

one_sample_z_prop(series=email_sessions['unsubscribed'], H0=0.008, alt='greater')

### Your Turn

1. Is the increase in unsubscribes statistically significant?
2. The email program beats the open-rate benchmark (Q3) but is increasing unsubscribes. What should FreshBrew do?

---
## Question 9: Mobile vs. Desktop Conversion Rates

> *"The product team says mobile converts differently than desktop. **Is that true?**"*

**Two-sample proportion test**. **Two-tailed** — just "are they different?"

### Step 1: Split into two groups

In [ ]:
# Filter by device, then pick the purchased column (0s and 1s)
mobile_purchases  = digital[digital['device'] == 'Mobile']['purchased']
desktop_purchases = digital[digital['device'] == 'Desktop']['purchased']

print("Mobile: ", mobile_purchases.sum(), "purchases out of", len(mobile_purchases))
print("Desktop:", desktop_purchases.sum(), "purchases out of", len(desktop_purchases))

### Step 2: Run the test

In [ ]:
two_sample_z_prop(series1=mobile_purchases, series2=desktop_purchases)

### Your Turn

1. Is the conversion rate difference statistically significant?
2. **Try it:** Do **loyalty members** purchase at a higher rate than **non-members**? Use the `loyalty_member` column to split and the `purchased` column to compare. Use `alt='greater'`.

In [ ]:
# YOUR CODE HERE
# Step 1: Filter digital to loyalty_member == True, pick 'purchased'
# Step 2: Filter digital to loyalty_member == False, pick 'purchased'
# Step 3: Run two_sample_z_prop with alt='greater'


---
## Question 10: Did the New Checkout Flow Improve Conversion? (Plug-in Numbers)

> *"The UX team tested a new checkout flow. Old flow: 200 purchases out of 5,000 sessions. New flow: 260 purchases out of 5,000 sessions. Is the new flow better?"*

Plug in the numbers. **Right-tailed** — we predict the new flow is *higher*.

In [ ]:
two_sample_z_prop(count1=260, n1=5000, count2=200, n2=5000, alt='greater')

### Your Turn

1. Is the new checkout flow significantly better?
2. What is the actual difference in conversion rates? Is that a meaningful improvement for the business?

---
# PART 3: Testing Correlation

> *"Now I want to ask about **relationships**. Not 'does X cause Y' — just 'are X and Y connected?'"*

---

## Question 11: Do Instagram Followers Predict Store Revenue?

> *"The social media manager says stores with more Instagram followers sell more. Is that actually true?"*

For a correlation test, we need **two numeric columns** from the same dataset.

In [ ]:
# Pick two columns and pass them to test_correlation
test_correlation(series1=operations['instagram_followers'],
                 series2=operations['monthly_revenue'])

### Your Turn

1. Is the correlation statistically significant?
2. The social media manager says "more followers **causes** more sales." What is wrong with this statement?
3. Look at r-squared in the output. What percentage of revenue variation is explained by Instagram followers?
4. **Try it:** Is there a correlation between `avg_ticket_usd` and `monthly_revenue` in the operations data?

In [ ]:
# YOUR CODE HERE
# Hint: test_correlation(series1=..., series2=...)


---
## Question 12: Did Price Increases Hurt Demand? (Plug-in Numbers)

> *"We tested price increases at 28 stores. The analyst reports r = -0.54 between price increase and change in units sold. Is that significant?"*

When you only have r and n from a report, plug them in directly.

In [ ]:
test_correlation(r=-0.54, n=28)

### Your Turn

1. What does r = -0.54 mean in plain English?
2. Look at r-squared. What percentage of the demand change is explained by price changes?
3. Should FreshBrew reverse the price increases?

---
## Question 13: Email Frequency and Revenue

> *"The VP claims email frequency drives revenue. r = 0.18 over 52 weeks. Is this significant?"*

In [ ]:
test_correlation(r=0.18, n=52)

### Your Turn

1. The VP insists: "r = 0.18 is positive, so more emails = more revenue!" How do you respond?
2. Even if this were significant, why can we not conclude that emails *cause* more revenue?

---
# PART 4: Now You Do the Full Analysis

For each challenge below, do the full workflow yourself:
1. Load the right dataset
2. Filter if needed
3. Pick the right column(s)
4. Choose and run the right test
5. Interpret the result

---

## Challenge A

> *"The satisfaction team says App Users have an average satisfaction score above 7.0. Is that true?"*

- Dataset: `freshbrew_survey.csv`
- Filter to: segment == 'App_User'
- Column: `satisfaction`
- Test: one-sample t-test, right-tailed, H0 = 7.0

In [49]:
# YOUR CODE HERE
# Filter to App Users
app_users = survey[survey['segment'] == 'App_User']

# Pick satisfaction column
app_sat = app_users['satisfaction']

print("App Users: n =", len(app_sat))
print("Mean satisfaction:", round(app_sat.mean(), 3))

# Right-tailed test: is mean > 7.0?
one_sample_t(series=app_sat, H0=7.0, alt='greater')

App Users: n = 223
Mean satisfaction: 7.258

ONE-SAMPLE T-TEST
H0: mean = 7.0
H1: mean > 7.0

Sample mean:  7.2583
Sample std:   1.7078
Sample size:  223
t-statistic:  2.2585
p-value:      0.0124

DECISION: Reject H0
  p-value 0.0124 < 0.05
  Evidence that mean > 7.0



{'t_stat': np.float64(2.2585303496449014),
 'p_value': np.float64(0.012442188892503038),
 'mean': np.float64(7.258295964125561),
 'std': 1.7078279662955262,
 'n': 223,
 'reject_H0': np.True_}

---
## Challenge B

> *"Do customers who pay with the App spend more than customers who pay with Cash?"*

- Dataset: `freshbrew_transactions.csv`
- Split by: `payment_method` (compare 'App' vs 'Cash')
- Column: `spend_usd`
- Test: two-sample t-test, right-tailed

In [50]:
# YOUR CODE HERE
# Split into two groups
app_spend  = transactions[transactions['payment_method'] == 'App']['spend_usd']
cash_spend = transactions[transactions['payment_method'] == 'Cash']['spend_usd']

print("App payment:  n =", len(app_spend),  " mean =", round(app_spend.mean(), 2))
print("Cash payment: n =", len(cash_spend), " mean =", round(cash_spend.mean(), 2))

# Right-tailed test: App > Cash
two_sample_t(series1=app_spend, series2=cash_spend, alt='greater')


App payment:  n = 680  mean = 6.08
Cash payment: n = 286  mean = 5.76

TWO-SAMPLE T-TEST
H0: mean1 - mean2 = 0
H1: mean1 - mean2 > 0

Group 1:  mean = 6.0754  std = 2.1612  n = 680
Group 2:  mean = 5.7628  std = 2.3366  n = 286
Difference:  0.3127
t-statistic: 1.9407
p-value:     0.0264

DECISION: Reject H0
  p-value 0.0264 < 0.05
  Evidence that mean1 - mean2 > 0



{'t_stat': np.float64(1.9407184900389824),
 'p_value': np.float64(0.0264271485932266),
 'mean1': np.float64(6.075441176470589),
 'mean2': np.float64(5.762762237762238),
 'diff': np.float64(0.31267893870835106),
 'reject_H0': np.True_}

---
## Challenge C

> *"Among Mobile users who are loyalty members, is the purchase rate above 8%?"*

- Dataset: `freshbrew_digital.csv`
- Filter to: `device == 'Mobile'` AND `loyalty_member == True`
- Column: `purchased` (0s and 1s)
- Test: one-sample proportion, right-tailed, H0 = 0.08

In [51]:
# YOUR CODE HERE
# Filter to Mobile AND loyalty_member == True
mobile_loyal = digital[
    (digital['device'] == 'Mobile') &
    (digital['loyalty_member'] == True)
]

purchased = mobile_loyal['purchased']

print("Sessions:", len(purchased))
print("Purchases:", purchased.sum())
print("Purchase rate:", round(purchased.mean(), 4))

# Right-tailed test: is rate > 0.08?
one_sample_z_prop(series=purchased, H0=0.08, alt='greater')


Sessions: 1515
Purchases: 130
Purchase rate: 0.0858

ONE-SAMPLE Z-TEST FOR PROPORTION
H0: proportion = 0.08
H1: proportion > 0.08

Successes:          130
Sample size:        1515
Sample proportion:  0.0858
z-statistic:        0.8334
p-value:            0.2023

DECISION: Fail to reject H0
  p-value 0.2023 >= 0.05
  Not enough evidence that proportion > 0.08



{'z_stat': np.float64(0.8333692057137688),
 'p_value': np.float64(0.20231826826475474),
 'p_hat': np.float64(0.0858085808580858),
 'n': 1515,
 'reject_H0': np.False_}

---
## Challenge D

> *"The CFO gave you summary stats from a competitor: Treatment group mean = 52, std = 12, n = 200. Control group mean = 48, std = 14, n = 180. Is there a significant difference?"*

- No dataset — use plug-in numbers
- Test: two-sample t-test, two-tailed

In [52]:
# YOUR CODE HERE
two_sample_t(mean1=52, std1=12, n1=200,
             mean2=48, std2=14, n2=180)



TWO-SAMPLE T-TEST
H0: mean1 - mean2 = 0
H1: mean1 - mean2 != 0

Group 1:  mean = 52  std = 12  n = 200
Group 2:  mean = 48  std = 14  n = 180
Difference:  4
t-statistic: 2.9741
p-value:     0.0031

DECISION: Reject H0
  p-value 0.0031 < 0.05
  Evidence that mean1 - mean2 != 0



{'t_stat': np.float64(2.974089582579658),
 'p_value': np.float64(0.003139814089945235),
 'mean1': 52,
 'mean2': 48,
 'diff': 4,
 'reject_H0': np.True_}

---
# Quick Reference

| What you want to test | Function | Data approach | Numbers approach |
|---|---|---|---|
| Mean vs. a benchmark | `one_sample_t()` | `series=df['col']` | `sample_mean=, sample_std=, n=` |
| Two group means | `two_sample_t()` | `series1=, series2=` | `mean1=, std1=, n1=, mean2=, ...` |
| Proportion vs. benchmark | `one_sample_z_prop()` | `series=df['col']` (0/1) | `count=, n=` |
| Two proportions | `two_sample_z_prop()` | `series1=, series2=` (0/1) | `count1=, n1=, count2=, n2=` |
| Correlation | `test_correlation()` | `series1=, series2=` | `r=, n=` |

**Options for all tests:**
- `alt='two-sided'` — "is it different?" (default)
- `alt='greater'` — "is it higher?"
- `alt='less'` — "is it lower?"
- `alpha=0.05` — significance level (default)

In [ ]:
# Run this anytime to see all available functions
show_help()